# S&P 500 Sector Analysis (2010–2020)

End-to-end risk-return profiling of 11 GICS sectors using sector ETFs (XLB, XLC, XLE, XLF, XLI, XLK, XLP, XLRE, XLU, XLV, XLY).

**Metrics:** Annualized Return · Annualized Volatility · Sharpe Ratio · Max Drawdown  
**Data source:** yfinance (adjusted close prices)  
**Period:** January 2010 – December 2020

In [ ]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
from src.data_loader import download_prices, compute_returns, save_to_db, load_returns
from src.metrics import sector_summary
from src.visualizations import plot_sector_returns, plot_risk_return_scatter, plot_correlation_heatmap

## 1. Load Data

Load daily returns from the SQLite database. If the database does not exist yet, the cell downloads data automatically (~30 seconds).

In [ ]:
try:
    returns = load_returns()
    print(f'Loaded {len(returns):,} rows from database.')
except Exception:
    print('Database not found — downloading data (~30 seconds)...')
    prices = download_prices()
    returns = compute_returns(prices)
    save_to_db(prices, returns)
    print(f'Done. Loaded {len(returns):,} rows.')

In [ ]:
print(f'Shape: {returns.shape}')
print('Date range:', returns['date'].min(), 'to', returns['date'].max())
print('Sectors:', sorted(returns['sector'].unique()))
returns.head(8)

## 2. Risk-Return Metrics

Compute annualized return, volatility, Sharpe ratio (risk-free rate = 2%), and max drawdown for each sector over the full 10-year period.

In [ ]:
summary = sector_summary(returns)

(
    summary.set_index('sector')
    .style
    .format({
        'annualized_return': '{:.2%}',
        'annualized_volatility': '{:.2%}',
        'sharpe_ratio': '{:.3f}',
        'max_drawdown': '{:.2%}',
    })
    .background_gradient(subset=['sharpe_ratio'], cmap='RdYlGn')
    .background_gradient(subset=['max_drawdown'], cmap='RdYlGn_r')
)

## 3. Visualizations

### 3.1 Annualized Return by Sector

In [ ]:
plot_sector_returns(summary).show()

### 3.2 Risk vs. Return

Bubble size represents Sharpe ratio. Upper-left quadrant = high return per unit of risk.

In [ ]:
plot_risk_return_scatter(summary).show()

### 3.3 Sector Return Correlation Matrix

Values near 1 indicate sectors that move together, reducing diversification benefit.

In [ ]:
returns_wide = returns.pivot_table(index='date', columns='sector', values='daily_return')
plot_correlation_heatmap(returns_wide).show()

## 4. Key Findings

| Sector | Takeaway |
|--------|----------|
| **Technology (XLK)** | Highest annualized return and Sharpe ratio — dominated the decade |
| **Energy (XLE)** | Worst performer; deep drawdowns driven by oil cycles and the 2020 demand collapse |
| **Health Care (XLV)** | Strong risk-adjusted returns with relatively low drawdown |
| **Utilities / Consumer Staples** | Lowest volatility; classic defensive characteristics |
| **Correlation structure** | Most sectors show moderate-to-high correlation (≥0.6), limiting intra-equity diversification |

**Takeaway:** The 2010–2020 period strongly favored growth sectors. A mean-variance efficient frontier would over-weight Technology and Health Care while minimizing Energy exposure.